Install OpenAI SDK

In [1]:
!pip -q install openai

Load Grok API Key from Colab Secret

In [4]:
!pip install groq
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [5]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


System Prompt

In [6]:
SYSTEM_PROMPT = """
You are an Apartment Maintenance AI Assistant.

Your responsibilities are:

1. Understand apartment maintenance complaints.
2. Identify issue category.
3. Assign priority:
   - Low
   - Medium
   - High
   - Critical
4. Suggest technician.
5. Estimate repair time.
6. Generate ticket summary.

Always return ONLY valid JSON.

Example format:

{
  "category":"",
  "priority":"",
  "technician":"",
  "estimated_time":"",
  "summary":""
}
"""

AI Function

In [9]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound
allam-2-7b
openai/gpt-oss-safeguard-20b
whisper-large-v3
whisper-large-v3-turbo
openai/gpt-oss-20b
meta-llama/llama-4-scout-17b-16e-instruct
qwen/qwen3-32b
llama-3.3-70b-versatile
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
qwen/qwen3.6-27b
llama-3.1-8b-instant
meta-llama/llama-prompt-guard-2-22m
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english


In [10]:
import json

SYSTEM_PROMPT = """
You are an Apartment Maintenance AI Assistant.

Your task is to:

1. Understand the maintenance complaint.
2. Identify the maintenance category.
3. Assign priority (Low, Medium, High, Critical).
4. Suggest the technician.
5. Estimate repair time.

Return ONLY valid JSON.

Example:

{
  "category":"Plumbing",
  "priority":"Medium",
  "technician":"Plumber",
  "estimated_time":"2 Hours",
  "summary":"Kitchen sink water leakage"
}
"""

def maintenance_ai(complaint):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":SYSTEM_PROMPT
            },
            {
                "role":"user",
                "content":complaint
            }
        ],
        temperature=0.2,
        response_format={"type": "json_object"}   # Force JSON output
    )

    return json.loads(response.choices[0].message.content)

In [11]:
complaint = "Water is leaking from my kitchen sink."

result = maintenance_ai(complaint)

print(result)

{'category': 'Plumbing', 'priority': 'Medium', 'technician': 'Plumber', 'estimated_time': '1.5 Hours', 'summary': 'Kitchen sink water leakage'}


Interactive Chat

In [12]:
while True:

    complaint = input("\nResident : ")

    if complaint.lower() == "exit":
        break

    result = maintenance_ai(complaint)

    print("\nAI Response")
    print(json.dumps(result, indent=4))


Resident : light is not glow in entrance

AI Response
{
    "category": "Electrical",
    "priority": "Low",
    "technician": "Electrician",
    "estimated_time": "30 Minutes",
    "summary": "Entrance light not working"
}

Resident : bed room power is not there last 1 hour

AI Response
{
    "category": "Electrical",
    "priority": "High",
    "technician": "Electrician",
    "estimated_time": "1 Hour",
    "summary": "Bedroom power outage"
}

Resident : water leakage in kitchen

AI Response
{
    "category": "Plumbing",
    "priority": "High",
    "technician": "Plumber",
    "estimated_time": "1.5 Hours",
    "summary": "Kitchen water leakage"
}

Resident : exit
